In [0]:
# Databricks notebook source
# =============================================================================
# HRMS WORK EXPERIENCE — DATA CLEANING PIPELINE
# Source   : hackathon_ltm.bronze.hrms_work_experience
# Silver   : hackathon_ltm.silver.silver_work_experience
# Quarantine: hackathon_ltm.quarantine.quarantine_work_experience
# =============================================================================

# COMMAND ----------
# STEP 0 — Setup & imports
# =============================================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DateType, IntegerType, StringType
from datetime import date

print("✅ STEP 0 — Libraries imported successfully")

# COMMAND ----------
# STEP 1 — Read source Delta table from Bronze layer
# =============================================================================
# Read the raw data as-is; no filters yet.
SOURCE_TABLE      = "hackathon_ltm.bronze.hrms_work_experience"
SILVER_TABLE      = "hackathon_ltm.silver.silver_work_experience"
QUARANTINE_TABLE  = "hackathon_ltm.quarantine.quarantine_work_experience"

df_raw = spark.read.table(SOURCE_TABLE)

print(f"✅ STEP 1 — Source table read successfully")
print(f"   Total records loaded : {df_raw.count()}")
print(f"   Schema               :")
df_raw.printSchema()
df_raw.show(5, truncate=False)

# COMMAND ----------
# STEP 2 — Add audit metadata column to track every record's origin
# =============================================================================
# We attach a processing timestamp and a mutable 'quarantine_reason' column
# that accumulates all failure reasons, pipe-separated.
df = (
    df_raw
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("quarantine_reason",   F.lit(None).cast(StringType()))
)

print("✅ STEP 2 — Audit columns added (ingestion_timestamp, quarantine_reason)")

# COMMAND ----------
# STEP 3 — Standardise column values (before validation)
# =============================================================================
# 3a. Trim leading/trailing whitespace from all string columns
string_cols = ["experience_id", "employee_id", "company_name", "job_role",
               "start_date", "end_date"]

for col in string_cols:
    df = df.withColumn(col, F.trim(F.col(col)))

# 3b. Normalise company_name casing — title-case after upper-trim
#     e.g. 'tcs' → 'TCS', 'infosys' → 'Infosys'
#     We handle known abbreviations explicitly to keep 'TCS', 'LTIMindtree' etc.
company_name_map = {
    "tcs"        : "TCS",
    "infosys"    : "Infosys",
    "wipro"      : "Wipro",
    "accenture"  : "Accenture",
    "cognizant"  : "Cognizant",
    "capgemini"  : "Capgemini",
    "ltimindtree": "LTIMindtree",
    "fresher"    : "Fresher",
}
mapping_expr = F.lower(F.col("company_name"))
for raw_val, clean_val in company_name_map.items():
    mapping_expr = F.when(F.lower(F.col("company_name")) == raw_val,
                          F.lit(clean_val)).otherwise(mapping_expr)
# Fall back to title-case for anything not in map
df = df.withColumn(
    "company_name",
    F.when(F.lower(F.col("company_name")).isin(list(company_name_map.keys())),
           mapping_expr)
     .otherwise(F.initcap(F.col("company_name")))
)

# 3c. Standardise job_role — 'Systems Engineer' → 'System Engineer'
#     (same role, minor naming drift)
df = df.withColumn(
    "job_role",
    F.when(F.col("job_role") == "Systems Engineer", F.lit("System Engineer"))
     .otherwise(F.col("job_role"))
)

print("✅ STEP 3 — Standardisation complete")
print("   3a. Whitespace trimmed for all string columns")
print("   3b. company_name case-normalised (e.g. 'tcs' → 'TCS')")
print("   3c. job_role alias resolved ('Systems Engineer' → 'System Engineer')")

# COMMAND ----------
# STEP 4 — Cast date columns to DateType
# =============================================================================
df = (
    df
    .withColumn("start_date", F.to_date(F.col("start_date"), "yyyy-MM-dd"))
    .withColumn("end_date",   F.to_date(F.col("end_date"),   "yyyy-MM-dd"))
)

print("✅ STEP 4 — Dates cast to DateType (yyyy-MM-dd)")

# COMMAND ----------
# STEP 5 — NULL / blank validation → quarantine
# =============================================================================
# Any record where a mandatory field is null or empty goes to quarantine.
mandatory_cols = ["experience_id", "employee_id", "company_name",
                  "job_role", "start_date", "end_date"]

null_condition = F.lit(False)
null_reason_expr = F.lit("")

for c in mandatory_cols:
    is_null = F.col(c).isNull() | (F.col(c).cast(StringType()) == "")
    null_condition = null_condition | is_null
    null_reason_expr = F.when(
        is_null,
        F.concat_ws("", null_reason_expr, F.lit(f"NULL_{c.upper()}|"))
    ).otherwise(null_reason_expr)

df = df.withColumn(
    "quarantine_reason",
    F.when(
        null_condition,
        F.concat(F.coalesce(F.col("quarantine_reason"), F.lit("")), null_reason_expr)
    ).otherwise(F.col("quarantine_reason"))
)

null_count = df.filter(null_condition).count()
print(f"✅ STEP 5 — NULL check complete  |  Flagged records : {null_count}")

# COMMAND ----------
# STEP 6 — Date logic validation → quarantine
# =============================================================================
# 6a. start_date cannot be after end_date
date_order_flag = F.col("start_date") > F.col("end_date")

df = df.withColumn(
    "quarantine_reason",
    F.when(
        date_order_flag,
        F.concat_ws("|",
                    F.coalesce(F.col("quarantine_reason"), F.lit("")),
                    F.lit("START_DATE_AFTER_END_DATE"))
    ).otherwise(F.col("quarantine_reason"))
)

# 6b. start_date should not be in the future
today_lit = F.lit(date.today()).cast(DateType())
future_start_flag = F.col("start_date") > today_lit

df = df.withColumn(
    "quarantine_reason",
    F.when(
        future_start_flag,
        F.concat_ws("|",
                    F.coalesce(F.col("quarantine_reason"), F.lit("")),
                    F.lit("FUTURE_START_DATE"))
    ).otherwise(F.col("quarantine_reason"))
)

# 6c. end_date should not be in the future
future_end_flag = F.col("end_date") > today_lit

df = df.withColumn(
    "quarantine_reason",
    F.when(
        future_end_flag,
        F.concat_ws("|",
                    F.coalesce(F.col("quarantine_reason"), F.lit("")),
                    F.lit("FUTURE_END_DATE"))
    ).otherwise(F.col("quarantine_reason"))
)

date_issues = df.filter(date_order_flag | future_start_flag | future_end_flag).count()
print(f"✅ STEP 6 — Date validation complete  |  Flagged records : {date_issues}")
print("   6a. start_date > end_date check")
print("   6b. Future start_date check")
print("   6c. Future end_date check")

# COMMAND ----------
# STEP 7 — Format / pattern validation → quarantine
# =============================================================================
# 7a. experience_id must match EXP followed by digits
exp_id_pattern = r'^EXP\d+$'
invalid_exp_id = ~F.col("experience_id").rlike(exp_id_pattern)

df = df.withColumn(
    "quarantine_reason",
    F.when(
        invalid_exp_id,
        F.concat_ws("|",
                    F.coalesce(F.col("quarantine_reason"), F.lit("")),
                    F.lit("INVALID_EXPERIENCE_ID_FORMAT"))
    ).otherwise(F.col("quarantine_reason"))
)

# 7b. employee_id must match E followed by digits
emp_id_pattern = r'^E\d+$'
invalid_emp_id = ~F.col("employee_id").rlike(emp_id_pattern)

df = df.withColumn(
    "quarantine_reason",
    F.when(
        invalid_emp_id,
        F.concat_ws("|",
                    F.coalesce(F.col("quarantine_reason"), F.lit("")),
                    F.lit("INVALID_EMPLOYEE_ID_FORMAT"))
    ).otherwise(F.col("quarantine_reason"))
)

format_issues = df.filter(invalid_exp_id | invalid_emp_id).count()
print(f"✅ STEP 7 — Format validation complete  |  Flagged records : {format_issues}")
print("   7a. experience_id pattern check: ^EXP\\d+$")
print("   7b. employee_id pattern check  : ^E\\d+$")

# COMMAND ----------
# STEP 8 — Duplicate detection → quarantine
# =============================================================================
# A true duplicate is same employee_id + company_name + start_date + end_date.
# Keep the first occurrence (by experience_id sort); quarantine the rest.
dup_window = Window.partitionBy(
    "employee_id", "company_name", "start_date", "end_date"
).orderBy("experience_id")

df = df.withColumn("_dup_rank", F.row_number().over(dup_window))

df = df.withColumn(
    "quarantine_reason",
    F.when(
        F.col("_dup_rank") > 1,
        F.concat_ws("|",
                    F.coalesce(F.col("quarantine_reason"), F.lit("")),
                    F.lit("DUPLICATE_RECORD"))
    ).otherwise(F.col("quarantine_reason"))
)

dup_count = df.filter(F.col("_dup_rank") > 1).count()
print(f"✅ STEP 8 — Duplicate detection complete  |  Flagged records : {dup_count}")
print("   Duplicate key : employee_id + company_name + start_date + end_date")
print("   Strategy      : Keep first by experience_id; quarantine rest")

# COMMAND ----------
# STEP 9 — Overlapping employment period detection → quarantine
# =============================================================================
# For the same employee, two records overlap if one period starts before
# the other ends (excluding exact boundary touches).
# We use a self-join approach via window lag.
overlap_window = Window.partitionBy("employee_id").orderBy("start_date")

df = (
    df
    .withColumn("_prev_end_date",
                F.lag("end_date").over(overlap_window))
    .withColumn("_prev_exp_id",
                F.lag("experience_id").over(overlap_window))
)

overlap_flag = (
    F.col("_prev_end_date").isNotNull() &
    (F.col("start_date") < F.col("_prev_end_date"))
)

df = df.withColumn(
    "quarantine_reason",
    F.when(
        overlap_flag,
        F.concat_ws("|",
                    F.coalesce(F.col("quarantine_reason"), F.lit("")),
                    F.lit("OVERLAPPING_EMPLOYMENT_PERIOD"))
    ).otherwise(F.col("quarantine_reason"))
)

overlap_count = df.filter(overlap_flag).count()
print(f"✅ STEP 9 — Overlap detection complete  |  Flagged records : {overlap_count}")

# COMMAND ----------
# STEP 10 — 'Fresher' company handling
# =============================================================================
# company_name = 'Fresher' is not a real employer. It represents a candidate
# with no prior work experience. These records are VALID but are flagged with
# a business tag so downstream KPIs can exclude or handle them separately.
df = df.withColumn(
    "is_fresher_record",
    F.when(F.col("company_name") == "Fresher", F.lit(True))
     .otherwise(F.lit(False))
)

fresher_count = df.filter(F.col("is_fresher_record")).count()
print(f"✅ STEP 10 — Fresher records tagged (not quarantined)")
print(f"   Fresher record count : {fresher_count}")
print(f"   These are kept in Silver with is_fresher_record = True")

# COMMAND ----------
# STEP 11 — Derive KPI-ready computed columns (Silver enrichment)
# =============================================================================
# Only applied on records that will NOT be quarantined.

# 11a. tenure_in_days  : raw day count per job
# 11b. tenure_in_months: months (approx, for grouping)
# 11c. tenure_in_years : years (rounded to 1 decimal, for avg tenure KPI)
# 11d. is_current_employer : end_date = most recent end_date per employee
#       (proxy for current job; better replaced with actual flag from HR system)
# 11e. employment_type_category : classifies role as Intern/Trainee/Individual Contributor/Lead/Manager
# 11f. company_tenure_bucket : Short (<1yr), Medium (1-3 yrs), Long (>3 yrs)
# 11g. experience_sequence_no : ordinal rank of each job per employee by start_date
# 11h. total_prior_experience_days : cumulative days of experience before current record (per employee)
# 11i. gap_from_prev_job_days : days between end of previous job and start of this one

df = (
    df
    # 11a. tenure_in_days
    .withColumn("tenure_in_days",
                F.datediff(F.col("end_date"), F.col("start_date")).cast(IntegerType()))

    # 11b. tenure_in_months
    .withColumn("tenure_in_months",
                F.round(F.col("tenure_in_days") / 30.44, 1))

    # 11c. tenure_in_years
    .withColumn("tenure_in_years",
                F.round(F.col("tenure_in_days") / 365.25, 2))

    # 11d. start_year, start_month, end_year, end_month (useful for time-based KPIs)
    .withColumn("start_year",  F.year("start_date").cast(IntegerType()))
    .withColumn("start_month", F.month("start_date").cast(IntegerType()))
    .withColumn("end_year",    F.year("end_date").cast(IntegerType()))
    .withColumn("end_month",   F.month("end_date").cast(IntegerType()))

    # 11e. employment_type_category
    .withColumn("employment_type_category",
        F.when(F.lower(F.col("job_role")).rlike("intern|student intern"), F.lit("Intern"))
         .when(F.lower(F.col("job_role")).rlike("trainee|fresher"), F.lit("Trainee"))
         .when(F.lower(F.col("job_role")).rlike("manager|director|vp|president|chief|head"), F.lit("Manager"))
         .when(F.lower(F.col("job_role")).rlike("lead|senior|principal|architect|specialist"), F.lit("Senior/Lead"))
         .otherwise(F.lit("Individual Contributor"))
    )

    # 11f. company_tenure_bucket
    .withColumn("company_tenure_bucket",
        F.when(F.col("tenure_in_days") < 365,              F.lit("Short (<1 yr)"))
         .when(F.col("tenure_in_days").between(365, 1094),  F.lit("Medium (1-3 yrs)"))
         .otherwise(                                         F.lit("Long (>3 yrs)"))
    )
)

# 11g. experience_sequence_no — ordinal rank per employee
seq_window = Window.partitionBy("employee_id").orderBy("start_date")
df = df.withColumn("experience_sequence_no",
                   F.row_number().over(seq_window).cast(IntegerType()))

# 11h. total_prior_experience_days (cumulative sum of tenure before this record)
cum_window = Window.partitionBy("employee_id").orderBy("start_date").rowsBetween(
    Window.unboundedPreceding, -1
)
df = df.withColumn("total_prior_experience_days",
                   F.coalesce(F.sum("tenure_in_days").over(cum_window), F.lit(0))
                    .cast(IntegerType()))

# 11i. gap_from_prev_job_days (days between previous job end and this job start)
df = df.withColumn("gap_from_prev_job_days",
    F.when(
        F.col("_prev_end_date").isNotNull(),
        F.datediff(F.col("start_date"), F.col("_prev_end_date"))
    ).otherwise(F.lit(None).cast(IntegerType()))
)

print("✅ STEP 11 — Derived KPI columns added:")
print("   tenure_in_days, tenure_in_months, tenure_in_years")
print("   start_year, start_month, end_year, end_month")
print("   employment_type_category, company_tenure_bucket")
print("   experience_sequence_no, total_prior_experience_days, gap_from_prev_job_days")
print("   is_fresher_record")

# COMMAND ----------
# STEP 12 — Split into Silver (clean) and Quarantine datasets
# =============================================================================
quarantine_flag = F.col("quarantine_reason").isNotNull() & \
                  (F.col("quarantine_reason") != "")

# Drop internal helper columns before writing
internal_cols = ["_dup_rank", "_prev_end_date", "_prev_exp_id"]

df_clean = (
    df
    .filter(~quarantine_flag)
    .drop(*internal_cols, "quarantine_reason")
)

df_quarantine = (
    df
    .filter(quarantine_flag)
    .drop(*internal_cols)
    # Keep all original + audit columns in quarantine for investigation
    .select(
        "experience_id", "employee_id", "company_name", "job_role",
        "start_date", "end_date",
        "quarantine_reason", "ingestion_timestamp"
    )
)

clean_count      = df_clean.count()
quarantine_count = df_quarantine.count()
total            = clean_count + quarantine_count

print(f"✅ STEP 12 — Dataset split complete")
print(f"   Total records      : {total}")
print(f"   Clean (Silver)     : {clean_count}  ({round(clean_count/total*100,1)}%)")
print(f"   Quarantine         : {quarantine_count}  ({round(quarantine_count/total*100,1)}%)")
print()
print("--- Quarantine breakdown ---")
df_quarantine.groupBy("quarantine_reason").count().orderBy("count", ascending=False).show(truncate=False)

# COMMAND ----------
# STEP 13 — Write Silver table (clean data)
# =============================================================================
# Using Delta MERGE to be idempotent (safe for reruns).
# Primary key : experience_id

spark.sql(f"CREATE SCHEMA IF NOT EXISTS hackathon_ltm.silver")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
        experience_id               STRING,
        employee_id                 STRING,
        company_name                STRING,
        job_role                    STRING,
        start_date                  DATE,
        end_date                    DATE,
        tenure_in_days              INT,
        tenure_in_months            DOUBLE,
        tenure_in_years             DOUBLE,
        start_year                  INT,
        start_month                 INT,
        end_year                    INT,
        end_month                   INT,
        employment_type_category    STRING,
        company_tenure_bucket       STRING,
        experience_sequence_no      INT,
        total_prior_experience_days INT,
        gap_from_prev_job_days      INT,
        is_fresher_record           BOOLEAN,
        ingestion_timestamp         TIMESTAMP
    )
    USING DELTA
    COMMENT 'Cleaned and enriched work experience data'
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

df_clean.createOrReplaceTempView("silver_staging")

spark.sql(f"""
    MERGE INTO {SILVER_TABLE} AS target
    USING silver_staging AS source
    ON target.experience_id = source.experience_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"✅ STEP 13 — Silver table written: {SILVER_TABLE}")
print(f"   Records written : {clean_count}")

# COMMAND ----------
# STEP 14 — Write Quarantine table
# =============================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS hackathon_ltm.quarantine")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {QUARANTINE_TABLE} (
        experience_id        STRING,
        employee_id          STRING,
        company_name         STRING,
        job_role             STRING,
        start_date           DATE,
        end_date             DATE,
        quarantine_reason    STRING,
        ingestion_timestamp  TIMESTAMP
    )
    USING DELTA
    COMMENT 'Quarantined records from hrms_work_experience with failure reasons'
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

df_quarantine.createOrReplaceTempView("quarantine_staging")

spark.sql(f"""
    MERGE INTO {QUARANTINE_TABLE} AS target
    USING quarantine_staging AS source
    ON target.experience_id = source.experience_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"✅ STEP 14 — Quarantine table written: {QUARANTINE_TABLE}")
print(f"   Records written : {quarantine_count}")

# COMMAND ----------
# STEP 15 — Final validation & summary
# =============================================================================
print("=" * 70)
print("   PIPELINE SUMMARY — hrms_work_experience Cleaning")
print("=" * 70)
print(f"   Source   : {SOURCE_TABLE}")
print(f"   Silver   : {SILVER_TABLE}")
print(f"   Quarantine: {QUARANTINE_TABLE}")
print()

silver_final = spark.read.table(SILVER_TABLE)
quar_final   = spark.read.table(QUARANTINE_TABLE)

print(f"   Silver record count      : {silver_final.count()}")
print(f"   Quarantine record count  : {quar_final.count()}")
print()
print("   Silver schema:")
silver_final.printSchema()
print("   Sample Silver records:")
silver_final.show(5, truncate=False)
print()
print("   Quarantine breakdown:")
quar_final.groupBy("quarantine_reason").count().show(truncate=False)

print("✅ STEP 15 — Pipeline completed successfully")

# COMMAND ----------
# =============================================================================
# APPENDIX — KPI HINTS  (columns available for downstream use)
# =============================================================================
#
# KPI                                     | Column(s) to use
# ----------------------------------------|----------------------------------------
# Avg tenure per company                  | company_name, tenure_in_years
# Avg tenure by job category              | employment_type_category, tenure_in_months
# Total prior experience at join          | total_prior_experience_days (last record per emp)
# Attrition / job-hopping index           | experience_sequence_no (high = many jobs)
# Freshers vs experienced hire mix        | is_fresher_record
# Career progression (individual contrib→ | employment_type_category ordered by seq
#   lead/manager)                         | experience_sequence_no
# Hiring trends by year                   | start_year, company_name
# Avg gap between jobs                    | gap_from_prev_job_days
# Tenure distribution bucket              | company_tenure_bucket
# =============================================================================

✅ STEP 0 — Libraries imported successfully
✅ STEP 1 — Source table read successfully
   Total records loaded : 500
   Schema               :
root
 |-- experience_id: string (nullable = true)
 |-- employee_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- job_role: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_system: string (nullable = true)
 |-- _source_file: string (nullable = true)

+-------------+-----------+------------+------------------------+----------+----------+--------------------------+--------------+------------------------------------------------------------+
|experience_id|employee_id|company_name|job_role                |start_date|end_date  |_ingestion_timestamp      |_source_system|_source_file                                                |
+-------------+-----------+------------+------------------------+----------+